# RAG Retrieval Metrics Evaluation

## Setup
Run this first to generate gRPC stubs:
```bash
pip install grpcio grpcio-tools
cd rag/pkg/rag && python -m grpc_tools.protoc -I=../../api --python_out=. --grpc_python_out=. ../../api/rag.proto
```

In [2]:
import json
from tqdm import tqdm
import grpc
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Set
from datetime import datetime
import sys

# Add path to generated proto files
sys.path.insert(0, '../pkg/rag')

import rag_pb2
import rag_pb2_grpc

## Configuration

In [3]:
# RAG Service connection
RAG_HOST = 'localhost'
RAG_PORT = 50051

# Load evaluation dataset
DATASET_PATH = '../rag_eval_dataset.json'

# Comparison methods enum mapping
COMPARISON_METHODS = {
    'COSINE': rag_pb2.COMPARISON_METHOD_COSINE,
    'DOT': rag_pb2.COMPARISON_METHOD_DOT,
    'EUCLIDEAN': rag_pb2.COMPARISON_METHOD_EUCLIDEAN,
    'L1': rag_pb2.COMPARISON_METHOD_L1
}

# Parameter grid to test
PARAM_GRID = {
    'limit': [
        3,
        5,
        10,
    ],
    'similarity_threshold': [
        0.33,
        0.5,
        0.75,
    ],
    'comparison_method': ['COSINE', 'DOT', 'EUCLIDEAN', 'L1']
}

# Metric N values (cutoffs for P@N, R@N)
N_VALUES = [3, 5, 10]

## Load Dataset

In [3]:
# with open(DATASET_PATH, 'r', encoding='utf-8') as f:
#     dataset = json.load(f)

# print(f"Loaded {len(dataset)} queries")

## RAG Client

In [4]:
# Title to doc_id mapping from database
TITLE_TO_DOC_ID = {
    "Альфа стипендия": "822bd8de-7ee2-44d2-a1fa-eab1d20c08fe",
    "Базовая стипендия (ГАС)": "c4c3b273-4c8d-4fa7-bb21-260afa5e599d",
    "Вид магистратуры и совмещение с работой": "6af3bc02-9d10-4e7d-a341-7d019a81bd38",
    "Виды стипендий": "94acc660-fe66-40f0-bc7b-295ffca0a6d4",
    "ВКР как арт-проект": "1a87c462-9ec7-44ae-8820-c15610344963",
    "ВКР как бизнес-проект": "16c13543-eb3d-4c6b-9fc5-960bd99b9dd9",
    "ВКР как научная статья": "8e84ce13-143a-47c0-ac5c-b21cda706135",
    "вопросы к экзамену пирсии.pdf": "8e650961-a113-4203-b3d6-ca83fa14602d",
    "Для народов с севера, Сибири, ДВ": "f015ffe6-d252-4092-b21d-945a874f8e53",
    "Доступ к громозеке (Кластеру ПИРСИИ)": "1677f1c4-c4d6-4093-bbf5-034facfaae6d",
    "Именные стипендии": "de489421-b3d7-443a-adc1-99d8b4793aba",
    "Конкурс Портфолио": "50c65b67-491a-44e7-9f06-ff9f309dfc75",
    "Контакты преподавателей": "72400b0b-50fa-44f7-8c24-3785ac75fb46",
    "КУГ 2025-2026": "5b2ce390-65ca-4034-93bb-d98920fc0108",
    "Материальная поддержка студентам и аспирантам": "fb457d25-cdd2-4737-819f-720836dd49fa",
    "Ответы на вопросы по ГИА": "63a25a2d-d378-4d83-8f62-c3b4842378f7",
    "Отчисления": "63d99656-58ee-4db8-b8cd-9eddb7615166",
    "повышенная стипендия (ПГАС)": "22e7ac20-7eaf-442a-a357-5b643987fd13",
    "Полезные материалы для студентов выпускного курса": "d9a37344-1469-484c-9b84-526f296fda7e",
    "Получение документов об окончании Университета ИТМО": "419e61f3-1854-4307-8db6-29d9cad78341",
    "ППА (пересдача)": "d62378e5-8fc1-4418-bae3-6b6840e7952d",
    "практика": "e516f9f7-a806-4a1c-8ed3-226a1ab2b275",
    "Рабочий план ПирСИИ": "3eb7f236-a9fe-4f90-88f1-bf3375e46a4e",
    "Семестровый обмен внутри РФ": "cd80703a-8aa3-4809-9872-feddea809b98",
    "Семестровый обмен за границей": "ff3b020d-f1e6-4926-a63c-e9c47f1f087e",
    "Социальная стипендия": "8c00999e-06a9-43e0-9e42-9bb9d46c5b35",
    "Социальный проект как диплом": "bf9b4320-cf7b-4e07-a406-e56b7c9d934f",
    "Стипендии Сбера": "1200170d-4eb7-4a27-88fb-601220cc2bc2",
    "Стипендия аспирантам": "ee2095f6-1095-402f-aeff-54a3ce64ebf8",
    "Стипендия Потанина": "8232082b-c65e-49ee-be55-0e1bc0ac9f57",
    "Стипендия правительства СПБ": "4bf38b27-47d6-4377-9250-c4a970fb48c5",
    "Стипендия президента для обучения за рубежом": "bcea65ff-52e2-464f-8221-25b7a9d16669",
    "Стипендия президента и правительства": "73748d2d-fce3-4f77-97f9-b79f1a97cee1",
    "Стипендия \"Система\"": "bc8bbec9-f759-416f-88a5-3c0633032fdb",
    "Темы ВКР и научные руководители": "366426c7-228a-4d48-998d-398ec892f91a",
    "учебный план": "773b23cd-faed-4dda-b828-8fe8265cc992"
}


# Title to doc_id mapping from database
TITLE_TO_DOC_ID = {
    "Альфа стипендия": "822bd8de-7ee2-44d2-a1fa-eab1d20c08fe",
    "Базовая стипендия (ГАС)": "c4c3b273-4c8d-4fa7-bb21-260afa5e599d",
    "Вид магистратуры и совмещение с работой": "6af3bc02-9d10-4e7d-a341-7d019a81bd38",
    "Виды стипендий": "94acc660-fe66-40f0-bc7b-295ffca0a6d4",
    "ВКР как арт-проект": "1a87c462-9ec7-44ae-8820-c15610344963",
    "ВКР как бизнес-проект": "16c13543-eb3d-4c6b-9fc5-960bd99b9dd9",
    "ВКР как научная статья": "8e84ce13-143a-47c0-ac5c-b21cda706135",
    "вопросы к экзамену пирсии.pdf": "8e650961-a113-4203-b3d6-ca83fa14602d",
    "Для народов с севера, Сибири, ДВ": "f015ffe6-d252-4092-b21d-945a874f8e53",
    "Доступ к громозеке (Кластеру ПИРСИИ)": "1677f1c4-c4d6-4093-bbf5-034facfaae6d",
    "Именные стипендии": "de489421-b3d7-443a-adc1-99d8b4793aba",
    "Конкурс Портфолио": "50c65b67-491a-44e7-9f06-ff9f309dfc75",
    "Контакты преподавателей": "72400b0b-50fa-44f7-8c24-3785ac75fb46",
    "КУГ 2025-2026": "5b2ce390-65ca-4034-93bb-d98920fc0108",
    "Материальная поддержка студентам и аспирантам": "fb457d25-cdd2-4737-819f-720836dd49fa",
    "Ответы на вопросы по ГИА": "63a25a2d-d378-4d83-8f62-c3b4842378f7",
    "Отчисления": "63d99656-58ee-4db8-b8cd-9eddb7615166",
    "повышенная стипендия (ПГАС)": "22e7ac20-7eaf-442a-a357-5b643987fd13",
    "Полезные материалы для студентов выпускного курса": "d9a37344-1469-484c-9b84-526f296fda7e",
    "Получение документов об окончании Университета ИТМО": "419e61f3-1854-4307-8db6-29d9cad78341",
    "ППА (пересдача)": "d62378e5-8fc1-4418-bae3-6b6840e7952d",
    "практика": "e516f9f7-a806-4a1c-8ed3-226a1ab2b275",
    "Рабочий план ПирСИИ": "3eb7f236-a9fe-4f90-88f1-bf3375e46a4e",
    "Семестровый обмен внутри РФ": "cd80703a-8aa3-4809-9872-feddea809b98",
    "Семестровый обмен за границей": "ff3b020d-f1e6-4926-a63c-e9c47f1f087e",
    "Социальная стипендия": "8c00999e-06a9-43e0-9e42-9bb9d46c5b35",
    "Социальный проект как диплом": "bf9b4320-cf7b-4e07-a406-e56b7c9d934f",
    "Стипендии Сбера": "1200170d-4eb7-4a27-88fb-601220cc2bc2",
    "Стипендия аспирантам": "ee2095f6-1095-402f-aeff-54a3ce64ebf8",
    "Стипендия Потанина": "8232082b-c65e-49ee-be55-0e1bc0ac9f57",
    "Стипендия правительства СПБ": "4bf38b27-47d6-4377-9250-c4a970fb48c5",
    "Стипендия президента для обучения за рубежом": "bcea65ff-52e2-464f-8221-25b7a9d16669",
    "Стипендия президента и правительства": "73748d2d-fce3-4f77-97f9-b79f1a97cee1",
    'Стипендия "Система"': "bc8bbec9-f759-416f-88a5-3c0633032fdb",
    "Темы ВКР и научные руководители": "366426c7-228a-4d48-998d-398ec892f91a",
    "учебный план": "773b23cd-faed-4dda-b828-8fe8265cc992"
}

class RAGClient:
    def __init__(self, host, port):
        self.channel = grpc.insecure_channel(f'{host}:{port}')
        self.stub = rag_pb2_grpc.RagServiceStub(self.channel)

    def search(self, query, limit=10, similarity_threshold=0.0, comparison_method='COSINE'):
        method = COMPARISON_METHODS.get(comparison_method, rag_pb2.COMPARISON_METHOD_COSINE)
        request = rag_pb2.SearchRequest(
            query=query,
            limit=limit,
            similarity_threshold=similarity_threshold,
            comparison_method=method
        )
        response = self.stub.SearchDocuments(request)
        results = []
        for doc in response.results:
            doc_id = doc.metadata.get('doc_id', doc.title)
            results.append({
                'doc_id': doc_id,
                'title': doc.title,
                'score': doc.similarity_score
            })
        return results

    def close(self):
        self.channel.close()

client = RAGClient(RAG_HOST, RAG_PORT)

## Metrics Calculation

In [5]:
def precision_at_k(retrieved, relevant, k):
    if k == 0 or not retrieved:
        return 0.0
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & relevant) / len(retrieved_k)

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & relevant) / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for i, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / i
    return 0.0

def calculate_metrics(retrieved, relevant, n_values):
    metrics = {}
    for n in n_values:
        metrics[f'P@{n}'] = precision_at_k(retrieved, relevant, n)
        metrics[f'R@{n}'] = recall_at_k(retrieved, relevant, n)
    metrics['MRR'] = reciprocal_rank(retrieved, relevant)
    return metrics

In [6]:
# Define all datasets to evaluate
DATASETS = {
    'dataset_1': '../rag_eval_dataset_1.json',
    'dataset_2': '../rag_eval_dataset_2.json',
    'dataset_3': '../rag_eval_dataset_3.json',
}

def run_evaluation_for_dataset(dataset_path, dataset_name):
    """Run evaluation on a single dataset and return results."""
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    print(f"\n=== Evaluating {dataset_name}: {len(dataset)} queries ===")

    all_results = []

    config_combinations = [
        (limit, sim_thresh, method)
        for limit in PARAM_GRID['limit']
        for sim_thresh in PARAM_GRID['similarity_threshold']
        for method in PARAM_GRID['comparison_method']
    ]

    total_configs = len(config_combinations)
    for config_idx, (limit, sim_thresh, method) in enumerate(config_combinations):
        if config_idx % 9 == 0:  # Print progress every 9 configs
            print(f"  Progress: {config_idx}/{total_configs} configs done")

        params = {
            'limit': limit,
            'similarity_threshold': sim_thresh,
            'comparison_method': method
        }

        all_metrics = defaultdict(list)

        for item_idx, item in enumerate(dataset):
            query = item['query']
            relevant_docs = set(
                TITLE_TO_DOC_ID.get(doc, doc) for doc in item['docs']
            )

            try:
                retrieved = client.search(
                    query=query,
                    limit=limit,
                    similarity_threshold=sim_thresh,
                    comparison_method=method
                )
                retrieved_ids = [r['doc_id'] for r in retrieved]
            except Exception as e:
                print(f"  Error: {e}")
                retrieved_ids = []

            metrics = calculate_metrics(retrieved_ids, relevant_docs, N_VALUES)
            for k, v in metrics.items():
                all_metrics[k].append(v)

        avg_metrics = {k: sum(v) / len(v) for k, v in all_metrics.items()}
        result_row = {
            'dataset': dataset_name,
            'queries_count': len(dataset),
            **params,
            **avg_metrics,
        }
        all_results.append(result_row)

    print(f"  Finished {total_configs} configs for {dataset_name}")
    return pd.DataFrame(all_results)

In [12]:
# Run evaluation on all datasets
all_dataset_results = []
for dataset_name, dataset_path in DATASETS.items():
    try:
        df = run_evaluation_for_dataset(dataset_path, dataset_name)
        all_dataset_results.append(df)
    except Exception as e:
        print(f"Error loading {dataset_name}: {e}")

# Combine all results
combined_df = pd.concat(all_dataset_results, ignore_index=True)
print(f"\nTotal configurations tested: {len(combined_df)}")


=== Evaluating dataset_1: 60 queries ===
  Progress: 0/36 configs done
  Progress: 9/36 configs done
  Progress: 18/36 configs done
  Progress: 27/36 configs done
  Finished 36 configs for dataset_1

=== Evaluating dataset_2: 51 queries ===
  Progress: 0/36 configs done
  Progress: 9/36 configs done
  Progress: 18/36 configs done
  Progress: 27/36 configs done
  Finished 36 configs for dataset_2

=== Evaluating dataset_3: 55 queries ===
  Progress: 0/36 configs done
  Progress: 9/36 configs done
  Progress: 18/36 configs done
  Progress: 27/36 configs done
  Finished 36 configs for dataset_3

Total configurations tested: 108


In [13]:
# Compute weighted average across datasets
# Weight by number of queries in each dataset

# Get total queries per dataset
queries_per_dataset = combined_df.groupby('dataset')['queries_count'].first().to_dict()
total_queries = sum(queries_per_dataset.values())

print("Queries per dataset:")
for ds, cnt in queries_per_dataset.items():
    print(f"  {ds}: {cnt} queries")
print(f"  Total: {total_queries} queries")

# Compute weighted average metrics
metric_cols = ['P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR']

weighted_results = []
param_cols = ['limit', 'similarity_threshold', 'comparison_method']

# Get unique parameter combinations
param_combinations = combined_df[param_cols].drop_duplicates()

for _, params in param_combinations.iterrows():
    mask = (
        (combined_df['limit'] == params['limit']) &
        (combined_df['similarity_threshold'] == params['similarity_threshold']) &
        (combined_df['comparison_method'] == params['comparison_method'])
    )
    subset = combined_df[mask]
    
    weighted_row = {
        'limit': params['limit'],
        'similarity_threshold': params['similarity_threshold'],
        'comparison_method': params['comparison_method'],
    }
    
    for metric in metric_cols:
        # Weighted average: sum(metric * queries) / total_queries
        weighted_sum = 0
        for _, row in subset.iterrows():
            ds_weight = queries_per_dataset[row['dataset']] / total_queries
            weighted_sum += row[metric] * ds_weight
        weighted_row[metric] = weighted_sum
    
    weighted_results.append(weighted_row)

weighted_df = pd.DataFrame(weighted_results)
print(f"\nComputed weighted average for {len(weighted_df)} parameter combinations")

Queries per dataset:
  dataset_1: 60 queries
  dataset_2: 51 queries
  dataset_3: 55 queries
  Total: 166 queries

Computed weighted average for 36 parameter combinations


In [ ]:
# Calculate combined scores and find best configuration
weighted_df['F1@5'] = 2 * weighted_df['P@5'] * weighted_df['R@5'] / (weighted_df['P@5'] + weighted_df['R@5'] + 1e-10)
weighted_df['Combined'] = weighted_df['P@5'] * 0.4 + weighted_df['R@5'] * 0.3 + weighted_df['MRR'] * 0.3

display_cols = ['limit', 'similarity_threshold', 'comparison_method', 'P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR', 'F1@5', 'Combined']

print("=== WEIGHTED AVERAGE RESULTS ACROSS ALL DATASETS ===\n")
print(weighted_df[display_cols].sort_values("Combined", ascending=False).to_string(index=False))

# Save aggregated results to CSV
import os
output_dir = './evaluation_results'
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

=== WEIGHTED AVERAGE RESULTS ACROSS ALL DATASETS ===

 limit  similarity_threshold comparison_method      P@3      P@5     P@10      R@3      R@5     R@10      MRR     F1@5  Combined
     3                  0.33               DOT 0.275100 0.275100 0.275100 0.743876 0.743876 0.743876 0.740964 0.401659  0.555492
     3                  0.50               DOT 0.275100 0.275100 0.275100 0.743876 0.743876 0.743876 0.740964 0.401659  0.555492
     3                  0.75               DOT 0.275100 0.275100 0.275100 0.743876 0.743876 0.743876 0.740964 0.401659  0.555492
     3                  0.33         EUCLIDEAN 0.273092 0.273092 0.273092 0.743876 0.743876 0.743876 0.735944 0.399515  0.553183
     3                  0.50         EUCLIDEAN 0.273092 0.273092 0.273092 0.743876 0.743876 0.743876 0.735944 0.399515  0.553183
    10                  0.33            COSINE 0.277108 0.183133 0.103012 0.749900 0.817169 0.889859 0.756999 0.299210  0.545503
     3                  0.33               

In [ ]:
# Compact table: best config for each metric
metrics_to_find = ['Combined', 'F1@5', 'MRR', 'P@5', 'R@5']
best_rows = []
for metric in metrics_to_find:
    best_idx = weighted_df[metric].idxmax()
    best = weighted_df.loc[best_idx]
    best_rows.append({
        'Metric': metric,
        'limit': int(best['limit']),
        'sim_thresh': best['similarity_threshold'],
        'method': best['comparison_method'],
        'Value': round(best[metric], 4)
    })

best_df = pd.DataFrame(best_rows)
print(best_df.to_string(index=False))

# Save compact results
best_df.to_csv(f'{output_dir}/best_results_{timestamp}.csv', index=False)

  Metric  limit  sim_thresh method  Value
Combined      3        0.33    DOT 0.5555
    F1@5      3        0.33    DOT 0.4017
     MRR     10        0.33    DOT 0.7623
     P@5      3        0.33    DOT 0.2751
     R@5     10        0.33 COSINE 0.8172


In [16]:
# Summary: Best configurations
print("\n" + "="*60)
print("BEST CONFIGURATION (Weighted Average)")
print("="*60)

best_combined_idx = weighted_df['Combined'].idxmax()
best_combined = weighted_df.loc[best_combined_idx]

print(f"\nBest by Combined Score:")
print(f"  limit: {int(best_combined['limit'])}")
print(f"  similarity_threshold: {best_combined['similarity_threshold']}")
print(f"  comparison_method: {best_combined['comparison_method']}")
print(f"\nMetrics:")
for col in ['P@3', 'P@5', 'P@10', 'R@3', 'R@5', 'R@10', 'MRR', 'F1@5', 'Combined']:
    print(f"  {col}: {best_combined[col]:.4f}")

print("\n\nBest by F1@5:")
best_f1_idx = weighted_df['F1@5'].idxmax()
best_f1 = weighted_df.loc[best_f1_idx]
print(f"  limit: {int(best_f1['limit'])}, sim_thresh: {best_f1['similarity_threshold']}, method: {best_f1['comparison_method']}")
print(f"  F1@5: {best_f1['F1@5']:.4f}")

print("\n\nBest by MRR:")
best_mrr_idx = weighted_df['MRR'].idxmax()
best_mrr = weighted_df.loc[best_mrr_idx]
print(f"  limit: {int(best_mrr['limit'])}, sim_thresh: {best_mrr['similarity_threshold']}, method: {best_mrr['comparison_method']}")
print(f"  MRR: {best_mrr['MRR']:.4f}")


BEST CONFIGURATION (Weighted Average)

Best by Combined Score:
  limit: 3
  similarity_threshold: 0.33
  comparison_method: DOT

Metrics:
  P@3: 0.2751
  P@5: 0.2751
  P@10: 0.2751
  R@3: 0.7439
  R@5: 0.7439
  R@10: 0.7439
  MRR: 0.7410
  F1@5: 0.4017
  Combined: 0.5555


Best by F1@5:
  limit: 3, sim_thresh: 0.33, method: DOT
  F1@5: 0.4017


Best by MRR:
  limit: 10, sim_thresh: 0.33, method: DOT
  MRR: 0.7623


In [17]:
# Per-dataset breakdown
print("\n" + "="*60)
print("PER-DATASET SUMMARY")
print("="*60)

for dataset_name in DATASETS.keys():
    ds_df = combined_df[combined_df['dataset'] == dataset_name].copy()
    ds_df['F1@5'] = 2 * ds_df['P@5'] * ds_df['R@5'] / (ds_df['P@5'] + ds_df['R@5'] + 1e-10)
    
    best_idx = ds_df['F1@5'].idxmax()
    best = ds_df.loc[best_idx]
    
    print(f"\n{dataset_name} (best F1@5):")
    print(f"  limit={int(best['limit'])}, sim_thresh={best['similarity_threshold']}, method={best['comparison_method']}")
    print(f"  P@5={best['P@5']:.4f}, R@5={best['R@5']:.4f}, F1@5={best['F1@5']:.4f}, MRR={best['MRR']:.4f}")


PER-DATASET SUMMARY

dataset_1 (best F1@5):
  limit=3, sim_thresh=0.33, method=DOT
  P@5=0.3000, R@5=0.7719, F1@5=0.4321, MRR=0.8194

dataset_2 (best F1@5):
  limit=3, sim_thresh=0.33, method=L1
  P@5=0.2810, R@5=0.7680, F1@5=0.4115, MRR=0.7222

dataset_3 (best F1@5):
  limit=3, sim_thresh=0.33, method=DOT
  P@5=0.2485, R@5=0.7091, F1@5=0.3680, MRR=0.6667


# доверительные интервалы

In [9]:
def run_evaluation_for_dataset(dataset_path, dataset_name):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    query_level_results = []

    config_combinations = [
        (limit, sim_thresh, method)
        for limit in PARAM_GRID['limit']
        for sim_thresh in PARAM_GRID['similarity_threshold']
        for method in PARAM_GRID['comparison_method']
    ]

    total_configs = len(config_combinations)

    print(f"\n=== Dataset {dataset_name}: {len(dataset)} queries ===")

    for config_idx, (limit, sim_thresh, method) in enumerate(config_combinations):

        if config_idx % 9 == 0:
            print(f"Progress: {config_idx}/{total_configs}")

        for item_idx, item in enumerate(dataset):

            query = item['query']
            relevant_docs = set(
                TITLE_TO_DOC_ID.get(doc, doc) for doc in item['docs']
            )

            try:
                retrieved = client.search(
                    query=query,
                    limit=limit,
                    similarity_threshold=sim_thresh,
                    comparison_method=method
                )
                retrieved_ids = [r['doc_id'] for r in retrieved]
            except Exception:
                retrieved_ids = []

            metrics = calculate_metrics(
                retrieved_ids,
                relevant_docs,
                N_VALUES
            )

            query_level_results.append({
                "dataset": dataset_name,
                "query_id": item_idx,
                "limit": limit,
                "similarity_threshold": sim_thresh,
                "comparison_method": method,
                **metrics
            })

    return pd.DataFrame(query_level_results)


all_dataset_results = []

for dataset_name, dataset_path in DATASETS.items():
    df = run_evaluation_for_dataset(dataset_path, dataset_name)
    all_dataset_results.append(df)

query_df = pd.concat(all_dataset_results, ignore_index=True)

import numpy as np

def bootstrap_ci_queries(values, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)

    values = np.array(values)
    n = len(values)

    boot_means = np.empty(n_boot)

    for i in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot_means[i] = sample.mean()

    return (
        values.mean(),
        np.percentile(boot_means, 2.5),
        np.percentile(boot_means, 97.5)
    )

group_cols = [
    "limit",
    "similarity_threshold",
    "comparison_method",
    "dataset"
]

metrics = ["P@5", "P@10", "R@5", "R@10", "MRR"]

results = []

for keys, group in query_df.groupby(group_cols):

    row = dict(zip(group_cols, keys))

    for metric in metrics:
        mean, low, high = bootstrap_ci_queries(group[metric].values)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci_low"] = low
        row[f"{metric}_ci_high"] = high

    results.append(row)

final_df = pd.DataFrame(results)

best_rows = []

for metric in ["MRR", "P@5", "R@5"]:
    best = final_df.loc[final_df[f"{metric}_mean"].idxmax()]

    best_rows.append({
        "metric": metric,
        "limit": best["limit"],
        "threshold": best["similarity_threshold"],
        "method": best["comparison_method"],
        "value": best[f"{metric}_mean"],
        "ci": (
            best[f"{metric}_ci_low"],
            best[f"{metric}_ci_high"]
        )
    })

best_df = pd.DataFrame(best_rows)
print(best_df)


=== Dataset dataset_1: 60 queries ===
Progress: 0/36
Progress: 9/36
Progress: 18/36
Progress: 27/36

=== Dataset dataset_2: 51 queries ===
Progress: 0/36
Progress: 9/36
Progress: 18/36
Progress: 27/36

=== Dataset dataset_3: 55 queries ===
Progress: 0/36
Progress: 9/36
Progress: 18/36
Progress: 27/36
  metric  limit  threshold     method     value  \
0    MRR     10       0.33  EUCLIDEAN  0.848519   
1    P@5      3       0.33        DOT  0.300000   
2    R@5      5       0.33         L1  0.852941   

                                          ci  
0   (0.7624027777777777, 0.9218518518518518)  
1  (0.26666666666666666, 0.3333333333333332)  
2   (0.7549019607843137, 0.9313725490196079)  


In [12]:
final_df.to_csv(f'final_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv', index=False)

In [13]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict


def run_evaluation_for_dataset(dataset_path, dataset_name):
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    rows = []

    config_combinations = [
        (limit, sim_thresh, method)
        for limit in PARAM_GRID['limit']
        for sim_thresh in PARAM_GRID['similarity_threshold']
        for method in PARAM_GRID['comparison_method']
    ]

    print(f"\n=== Dataset {dataset_name}: {len(dataset)} queries ===")

    for config_idx, (limit, sim_thresh, method) in enumerate(config_combinations):

        if config_idx % 9 == 0:
            print(f"Progress: {config_idx}/{len(config_combinations)}")

        for item_idx, item in enumerate(dataset):

            query = item['query']
            relevant_docs = set(
                TITLE_TO_DOC_ID.get(doc, doc) for doc in item['docs']
            )

            try:
                retrieved = client.search(
                    query=query,
                    limit=limit,
                    similarity_threshold=sim_thresh,
                    comparison_method=method
                )
                retrieved_ids = [r['doc_id'] for r in retrieved]
            except Exception:
                retrieved_ids = []

            metrics = calculate_metrics(
                retrieved_ids,
                relevant_docs,
                N_VALUES
            )

            rows.append({
                # identity
                "dataset": dataset_name,
                "query_id": item_idx,

                # config params (IMPORTANT: extendable later)
                "limit": limit,
                "similarity_threshold": sim_thresh,
                "comparison_method": method,

                # metrics
                **metrics
            })

    return pd.DataFrame(rows)

In [ ]:
all_dataset_results = []

for dataset_name, dataset_path in DATASETS.items():
    df = run_evaluation_for_dataset(dataset_path, dataset_name)
    all_dataset_results.append(df)

query_df = pd.concat(all_dataset_results, ignore_index=True)


=== Dataset dataset_1: 60 queries ===
Progress: 0/36


In [ ]:
def bootstrap_ci(values, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)

    values = np.array(values)
    n = len(values)

    boot = np.empty(n_boot)

    for i in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot[i] = sample.mean()

    return values.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5)

In [ ]:
group_cols = [
    "limit",
    "similarity_threshold",
    "comparison_method",
    "dataset"
]

metrics = ["P@5", "P@10", "R@5", "R@10", "MRR"]

results = []

for keys, group in query_df.groupby(group_cols):

    row = dict(zip(group_cols, keys))

    for metric in metrics:
        mean, low, high = bootstrap_ci(group[metric].values)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci_low"] = low
        row[f"{metric}_ci_high"] = high

    results.append(row)

final_df = pd.DataFrame(results)
final_df.to_csv(f'final_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv', index=False)

## Close Connection

In [19]:
client.close()
print("Done!")

Done!
